In [ ]:
import pandas as pd
import re
from src.inputs.capacity_mappings import MASTER_UNITS

def extract_capacity_info(text):
    """
    Extracts numeric value and unit from capacity string.
    
    Args:
        text (str): Capacity string, e.g., "5.2 GWh".
    
    Returns:
        tuple: (value, unit), or (None, None) if extraction fails.
    """
    if not isinstance(text, str):
        return None, None
    
    # Regex to extract numeric value and unit
    match = re.search(r"([0-9,.]+)\s*([A-Za-z]+)", text)
    if match:
        value = float(match.group(1).replace(",", ""))
        unit = match.group(2)
        return value, unit
    return None, None

def normalize_capacity(row):
    """
    Normalize capacity value to the master unit based on the component.
    
    Args:
        row (pd.Series): A row containing 'capacity' and 'component' columns.
    
    Returns:
        float: Capacity normalized to the master unit, or None if conversion fails.
    """
    value, unit = extract_capacity_info(row['capacity'])
    component = row['component']
    
    if pd.notna(value) and component in MASTER_UNITS:
        scaling = MASTER_UNITS[component]['scaling']
        if unit in scaling:
            return value * scaling[unit]  # Convert to master unit
    return None

def process_capacity_column(df):
    """
    Processes the capacity column and normalizes values to master units.

    Args:
        df (pd.DataFrame): Input DataFrame with 'capacity' and 'component'.

    Returns:
        pd.DataFrame: DataFrame with normalized capacity column 'capacity_normalized'.
    """
    # Extract numeric value and unit for reference
    df[['capacity_value', 'capacity_unit']] = df['capacity'].apply(
        lambda x: pd.Series(extract_capacity_info(x))
    )

    # Normalize capacity to master units
    df['capacity_normalized'] = df.apply(normalize_capacity, axis=1)
    
    # Add master unit column for clarity
    df['master_unit'] = df['component'].map(lambda x: MASTER_UNITS[x]['unit'] if x in MASTER_UNITS else None)
    return df

#START!!!!

In [14]:
import pandas as pd
import openpyxl
from quantulum3 import parser
import re

In [72]:

import pandas as pd
import re
from word2number import w2n

# Load your Excel
file_path = 'C:/Users/marie.juge/OneDrive - Bruegel/Desktop/tuone/tuone_v6/reconcile/storage/output/clean_output_ben.xlsx'
df = pd.read_excel(file_path)
df = df[["capacity"]].copy()
df["capacity"] = df["capacity"].fillna("")

# Define scale word → numeric multiplier
scale_map = {
    "thousand": 1e3,
    "million": 1e6,
    "billion": 1e9,
    "trillion": 1e12
}

# Main parser
def extract_capacity_info(text):
    """
    Returns:
        value (float): numeric part
        scale (float): multiplier like 1000000 for 'million'
        text (str): remaining string
    """
    if not isinstance(text, str):
        return None, None, None

    text = text.strip()

    # Case 1: 1.2 million EVs
    match_num = re.match(r"^([0-9,.]+)\s+(thousand|million|billion|trillion)\b\s*(.*)", text, re.IGNORECASE)
    if match_num:
        value = float(match_num.group(1).replace(",", ""))
        scale = scale_map.get(match_num.group(2).lower())
        remaining = match_num.group(3).strip()
        return value, scale, remaining

    # Case 2: one million EVs
    match_word = re.match(r"^([a-z\s-]+)\s+(thousand|million|billion|trillion)\b\s*(.*)", text, re.IGNORECASE)
    if match_word:
        try:
            value = w2n.word_to_num(match_word.group(1).strip())
            scale = scale_map.get(match_word.group(2).lower())
            remaining = match_word.group(3).strip()
            return value, scale, remaining
        except:
            pass

    # Case 3: simple number + unit like "5000 tonnes"
    match_plain = re.match(r"([0-9,.]+)\s+(.*)", text)
    if match_plain:
        return float(match_plain.group(1).replace(",", "")), None, match_plain.group(2).strip()

    return None, None, text

# Apply to dataframe
df[["value", "scale", "text"]] = df["capacity"].apply(lambda x: pd.Series(extract_capacity_info(x)))

# Show result
print(df.head(10))



# Define the time mapping
time_map = {
    "yearly": "per year",
    "/year": "per year",
    "annually": "per year",
    "per annum": "per year",
    "per year": "per year",
    "a year": "per year",
    "monthly": "per month",
    "per month": "per month",
    "/month": "per month",
    "a month": "per month",
    "weekly": "per week",
    "per week": "per week",
    "/week": "per week",
    "daily": "per day",
    "per day": "per day",
    "/day": "per day",
    "a day": "per day",
    "hourly": "per hour",
    "per hour": "per hour",
    "/hour": "per hour",
    "an hour": "per hour"
}

# Function to extract and remove time unit from text
def extract_normalized_time_unit(text):
    if not isinstance(text, str):
        return None, text

    text_lower = text.lower()
    for pattern, replacement in time_map.items():
        if re.search(rf"\b{re.escape(pattern)}\b", text_lower):
            # Remove the matched time phrase from the original text
            cleaned_text = re.sub(rf"\b{re.escape(pattern)}\b", "", text, flags=re.IGNORECASE).strip()
            return replacement, cleaned_text

    return None, text


# Apply the function and store results in two new columns
df[["time", "text"]] = df["text"].apply(
    lambda x: pd.Series(extract_normalized_time_unit(x))
)

print(df)


import re

metric_map = {
    "gwh": "gigawatt hour",
    "mwh": "megawatt hour",
    "kwh": "kilowatt hour",
    "twh": "terawatt hour",
    "wh": "watt hour",
    
    "mw": "megawatt",
    "kw": "kilowatt",
    "tw": "terawatt",

    "tonne": "tonne",
    "tonnes": "tonne",
    "mt": "megatonne",
    "kt": "kilotonne",
    "kg": "kilogram",
    "g": "gram"
}
def extract_normalized_metric_unit(text):
    if not isinstance(text, str):
        return None, text

    text_lower = text.lower()
    for pattern, replacement in metric_map.items():
        if re.search(rf"\b{re.escape(pattern)}\b", text_lower):
            # Remove the matched unit from the original text
            cleaned_text = re.sub(rf"\b{re.escape(pattern)}\b", "", text, flags=re.IGNORECASE).strip()
            return replacement, cleaned_text

    return None, text
df[["metric", "text"]] = df["text"].apply(
    lambda x: pd.Series(extract_normalized_metric_unit(x))
)




                            capacity     value  scale  \
0                                          NaN    NaN   
1                60,000 EVs per year   60000.0    NaN   
2                    40 GWh per year      40.0    NaN   
3      1,000 electric buses per year    1000.0    NaN   
4               5,000 units per year    5000.0    NaN   
5  10,000 delivery vehicles per year   10000.0    NaN   
6                                          NaN    NaN   
7                 100 units per year     100.0    NaN   
8             30,000 tonnes per year   30000.0    NaN   
9             400,000 units per year  400000.0    NaN   

                         text  
0                              
1                EVs per year  
2                GWh per year  
3     electric buses per year  
4              units per year  
5  delivery vehicles per year  
6                              
7              units per year  
8             tonnes per year  
9              units per year  
                    

In [ ]:
import pandas as pd
import re
from word2number import w2n

# ========= Load Excel =========
def load_capacity_column(file_path):
    df = pd.read_excel(file_path)
    df = df[["capacity"]].copy()
    df["capacity"] = df["capacity"].fillna("")
    return df

# ========= Scale Extraction (e.g., one million EVs) =========
scale_map = {
    "thousand": 1e3,
    "million": 1e6,
    "billion": 1e9,
    "trillion": 1e12
}

def extract_capacity_info(text):
    """
    Extracts numeric value (float or list), scale (float), and remaining text from capacity string.
    Handles:
        - ranges: "600,000 to 700,000 cars"
        - scaled numbers: "1.2 million EVs"
        - number words: "one million EVs"
        - simple: "5,000 tonnes"
    """
    if not isinstance(text, str):
        return None, None, None

    text = text.strip()

    # Case 0: Range "600,000 to 700,000 cars"
    match_range = re.match(r"^([0-9,.]+)\s*(?:to|-|–|—)\s*([0-9,.]+)\s+(.*)", text, re.IGNORECASE)
    if match_range:
        try:
            val1 = float(match_range.group(1).replace(",", ""))
            val2 = float(match_range.group(2).replace(",", ""))
            remaining = match_range.group(3).strip()
            return [val1, val2], None, remaining
        except:
            pass

    # Case 1: 1.2 million EVs
    match_num = re.match(r"^([0-9,.]+)\s+(thousand|million|billion|trillion)\b\s*(.*)", text, re.IGNORECASE)
    if match_num:
        value = float(match_num.group(1).replace(",", ""))
        scale = scale_map.get(match_num.group(2).lower())
        remaining = match_num.group(3).strip()
        return value, scale, remaining

    # Case 2: one million EVs
    match_word = re.match(r"^([a-z\s-]+)\s+(thousand|million|billion|trillion)\b\s*(.*)", text, re.IGNORECASE)
    if match_word:
        try:
            value = w2n.word_to_num(match_word.group(1).strip())
            scale = scale_map.get(match_word.group(2).lower())
            remaining = match_word.group(3).strip()
            return value, scale, remaining
        except:
            pass

    # Case 3: simple numeric prefix like "5000 tonnes"
    match_plain = re.match(r"([0-9,.]+)\s+(.*)", text)
    if match_plain:
        return float(match_plain.group(1).replace(",", "")), None, match_plain.group(2).strip()

    return None, None, text


# ========= Time Unit Extraction =========
time_map = {
    "yearly": "per year", "/year": "per year", "annually": "per year", "per annum": "per year",
    "per year": "per year", "a year": "per year",
    "monthly": "per month", "per month": "per month", "/month": "per month", "a month": "per month",
    "weekly": "per week", "per week": "per week", "/week": "per week",
    "daily": "per day", "per day": "per day", "/day": "per day", "a day": "per day",
    "hourly": "per hour", "per hour": "per hour", "/hour": "per hour", "an hour": "per hour"
}

def extract_normalized_time_unit(text):
    """
    Extracts time frequency (e.g., per year) and cleans the text.
    """
    if not isinstance(text, str):
        return None, text

    text_lower = text.lower()
    for pattern, replacement in time_map.items():
        if re.search(rf"\b{re.escape(pattern)}\b", text_lower):
            cleaned_text = re.sub(rf"\b{re.escape(pattern)}\b", "", text, flags=re.IGNORECASE).strip()
            return replacement, cleaned_text
    return None, text

# ========= Metric Unit Extraction =========
metric_map = {
    "gwh": "gigawatt hour", "mwh": "megawatt hour", "kwh": "kilowatt hour", "twh": "terawatt hour", "wh": "watt hour",
    "mw": "megawatt", "kw": "kilowatt", "tw": "terawatt",
    "tonne": "tonne", "tonnes": "tonne", "tons": "tonne","mt": "megatonne", "kt": "kilotonne", "kg": "kilogram", "g": "gram"
}

def extract_normalized_metric_unit(text):
    """
    Extracts standardized energy or weight metric and removes it from the text.
    """
    if not isinstance(text, str):
        return None, text

    text_lower = text.lower()
    for pattern, replacement in metric_map.items():
        if re.search(rf"\b{re.escape(pattern)}\b", text_lower):
            cleaned_text = re.sub(rf"\b{re.escape(pattern)}\b", "", text, flags=re.IGNORECASE).strip()
            return replacement, cleaned_text
    return None, text

# ========= Run Full Extraction Pipeline =========
def run_extraction_pipeline(file_path):
    df = load_capacity_column(file_path)

    df[["value", "scale", "text"]] = df["capacity"].apply(lambda x: pd.Series(extract_capacity_info(x)))
    df[["time", "text"]] = df["text"].apply(lambda x: pd.Series(extract_normalized_time_unit(x)))
    df[["metric", "text"]] = df["text"].apply(lambda x: pd.Series(extract_normalized_metric_unit(x)))

    return df

# ========= Execute =========
file_path = 'C:/Users/marie.juge/OneDrive - Bruegel/Desktop/tuone/tuone_v6/reconcile/storage/output/clean_output_ben.xlsx'
df_result = run_extraction_pipeline(file_path)

# ========= Preview =========
print(df_result.head(10))




                            capacity     value  scale               text  \
0                                          NaN    NaN                      
1                60,000 EVs per year   60000.0    NaN                EVs   
2                    40 GWh per year      40.0    NaN                      
3      1,000 electric buses per year    1000.0    NaN     electric buses   
4               5,000 units per year    5000.0    NaN              units   
5  10,000 delivery vehicles per year   10000.0    NaN  delivery vehicles   
6                                          NaN    NaN                      
7                 100 units per year     100.0    NaN              units   
8             30,000 tonnes per year   30000.0    NaN                      
9             400,000 units per year  400000.0    NaN              units   

       time         metric  
0      None           None  
1  per year           None  
2  per year  gigawatt hour  
3  per year           None  
4  per year       